In [ ]:
import sys; sys.path.append('..')
sys.path.append('../curved_linesearch/')
import MeshFEM, mesh, mesh_energy, benchmark, viewer, py_newton_optimizer

import numpy as np
import igl

import matplotlib
from matplotlib import pyplot as plt

In [ ]:
import sim_utils, param_utils
import extra_utils, opt_utils

In [ ]:
from curved_linesearch import visualization

In [ ]:
import newton_flow
import newton_flow_utils as nfu

In [ ]:
import rotation_strain_extrapolation
from Benchmark import helper_funcs

# Newton Flow Problem and its settings

In [ ]:
model = 'cow2Disc.off'
model_name = 'cow'

In [ ]:
m = helper_funcs.read_mesh(f'../../../Models/TableOneModels/{model}')
print(f"Model: {model} Vertices: {m.numVertices()}")
print(f"Model: {model} Elements: {m.numElements()}")

In [ ]:
uv = mesh_energy.NodalVars(m, 2)
m_2d = mesh.Mesh(np.zeros((m.numVertices(),2)), m.elements())
m_2d.reembedElements(m.vertices())


m_init_2d = mesh.Mesh('ToysMesh/cow_init_2d.obj')
uv.setVars(m_init_2d.vertices().ravel())
nf = newton_flow.symmetric_dirichlet(m_2d, uv)

In [ ]:
nf.projectionSmoothingEpsilon = 0 # 1e-4 # 1e-8

In [ ]:
prob = py_newton_optimizer.NewtonMultiobjectiveProblem(uv, [nf])

In [ ]:
nf.elementHessianShift = 1e-11
prob.hessianShift = 0
prob.useRelativeHessianShift = False

In [ ]:
FIX_VARS = False
always_project = False

In [ ]:
opt = prob.optimizer()
opt.options.hessianProjectionController.startWithProjectionActive = False
opt.options.hessianProjectionController.numProjectionStepsBeforeDisable = 1
opt.options.hessianProjectionController.numConsecutiveIndefiniteStepsBeforeEnable = 0
if always_project: opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAlways()
opt.options.niter = 200

In [ ]:
x_init = prob.getVars()

In [ ]:
prob.energy()

## Flip avoiding linesearch

In [ ]:
# import flip_avoiding_step_length
# prob.initialFeasibleStepLengthComputer = flip_avoiding_step_length.FlipAvoidingStepLength(m_2d.elements())
# prob.initialFeasibleStepLengthComputer.backoffFactor = 0.95

In [ ]:
# fasl = flip_avoiding_step_length.FlipAvoidingStepLength(m_2d.elements())

# Construct Extrapolate Method Class

In [ ]:
linear_extrapolator = extra_utils.LinearExtrapolator()

In [ ]:
pade_extrapolator = extra_utils.PadeExtrapolator(opt, 14)

In [ ]:
RS_extrapolator = rotation_strain_extrapolation.RSNewtonFlowExtrapolator(m_2d)

In [ ]:
# brek

# Newton Optimize

In [ ]:
initial_iter_num = 5

In [ ]:
opt.options.niter = initial_iter_num

In [ ]:
benchmark.reset()
opt.optimize()
benchmark.report()


In [ ]:
# line_search_method = opt_utils.BruteForceLinesearch(alpha_step_size=0.01)
# benchmark.reset()
# vertices_list = opt_utils.newton_extrapolate(opt, linear_extrapolator, line_search_method, max_iters=10, verbose=True)
# benchmark.report()

In [ ]:
brutal_line_search = opt_utils.BruteForceLinesearch(alpha_step_size=0.01)
ternary_line_search = opt_utils.TernaryLinesearch()
golden_section_search = opt_utils.GoldenSectionSearch()
parabola_fit_search = opt_utils.ParabolaFitSearch()

In [ ]:
max_alpha = 10
brutal_line_search.max_alpha= max_alpha

ternary_line_search.max_alpha = max_alpha
ternary_line_search.ternary_tol = 1

golden_section_search.max_alpha = max_alpha
golden_section_search.golden_tol = 1

parabola_fit_search.max_alpha = max_alpha
parabola_fit_search.use_extrapolation = True

In [ ]:
line_search_method = brutal_line_search

In [ ]:
optimal_alpha_list = []
def customCallback(prob, iter_count, alpha):
    optimal_alpha_list.append(alpha)

In [ ]:
benchmark.reset()
vertices_list = opt_utils.newton_extrapolate(opt, RS_extrapolator, line_search_method, grad_tol=2e-8, max_iters=50, 
                                             verbose=True, newton_step_tol=5e-2, max_extraNewton_stop_counter=None)
# opt.optimize()
benchmark.report()

In [ ]:
# brek

In [ ]:
vertices_list.shape

In [ ]:
len(optimal_alpha_list)

In [ ]:
final_uv = vertices_list[-1]

In [ ]:
initial_uv = vertices_list[0]

# UV Viewer

In [ ]:
uv_final = mesh.Mesh(final_uv, m.elements())

In [ ]:
uv_viewer = viewer.Viewer(uv_final, wireframe=True)
uv_viewer.show()

In [ ]:
# i=5
# uv_viewer.update(mesh = mesh.Mesh(newton_vertices[i], m.elements()))

## Flow Visualization

In [ ]:
fv = vertices_list

In [ ]:
extrapolation_dist = max_alpha
constant_speed = True
num_frames = min(500, len(fv))

methods = [(1, nfu.eval_trajectory_taylor, 'Newton'),
           (2, nfu.eval_trajectory_taylor, 'Deg 2 Taylor'),
           (3, nfu.eval_trajectory_taylor, 'Deg 3 Taylor'),
           (1, RS_extrapolator, 'Poisson'),
           (14, nfu.eval_trajectory_vector_pade, 'Pade 14'),
           (19, nfu.eval_trajectory_vector_pade, 'Pade 19')
]

methods = [
    (1, nfu.eval_trajectory_taylor, 'Newton'),
    # (14, nfu.eval_trajectory_vector_pade, 'Pade 14'),
    (1, RS_extrapolator, 'Poisson'),
]


ff = lambda i:  visualization.flow_frame(i, opt, fv, extrapolation_dist, constant_speed,
                         extrapolation_method_list=methods, truncate=True, corners_only=True)

In [ ]:
# visualization.writeVideo('videos/Poisson_vis_traj.mp4',num_frames,ff)

## plot alpha functions

In [ ]:
def plot_optimal_alpha(axs, ind):
    plt.sca(axs[1])
    plt.axvline(x=optimal_alpha_list[ind], color='r', lw=1, ls='--')
    return plt

def plot_cached_alphas(axs, ind, pltOptOnly=True):
    plt.sca(axs[1])
    # line search configure
    line_search_func = line_search_method
    extrapolator = RS_extrapolator
    
    x = fv[ind].ravel()
    prob.setVars(x)
    d = opt.newton_step()
    f0 = prob.energy()
    df0 = np.dot(d, prob.gradient())
    
    
    # Linesearch prepare
    extrapolator.linesearch_begin(x, d)
    def f(alpha):
        x_new = extrapolator.linesearch_eval(alpha)
        o = prob.objectiveAtVars(x_new.ravel())
        return o
    optimal_alpha = line_search_func(f, x, d, f0, df0)
    
    # plot cached alphas
    if not pltOptOnly:
        alphas_cached = list(line_search_func.cache.keys())
        for alpha in alphas_cached:
            plt.axvline(x=alpha, color='#B07AA1', lw=1, ls='--')
    # plot optimal alpha
    plt.axvline(x=optimal_alpha, color='r', lw=1, ls='--')
    
    # prob.setVars to original?
    return plt

In [ ]:
i = 2
axs = ff(i)
# line_search begin call first, and copy f(alpha) and then do line_search eval
# plt.sca(axs[1])
# plt.axvline(x=4, color='r', lw=1, ls='--')
plt = plot_cached_alphas(axs, i, pltOptOnly=True)

In [ ]:
# ff(24)

In [ ]:
# brek

## Visualize Element Energy plot (roi's version)

In [ ]:
fv = vertices_list

In [ ]:
vis_indx = 7

In [ ]:
def prepare_alpha_eleEnergies(ind, alphas):
    x = fv[ind].ravel()
    prob.setVars(x)
    d = opt.newton_step()
    
    # Linesearch prepare
    RS_extrapolator.linesearch_begin(x, d)
    linear_extrapolator.linesearch_begin(x, d)
    
    def getEleEnergies(alpha, ex):
        x_new = ex.linesearch_eval(alpha)
        prob.setVars(x_new.ravel())
        ele_energies = [nf.elementEnergy(ei) for ei in range(nf.numElements())]
        # ele_energies.sort()
        return ele_energies
    
    alpha_eleEnergies = [(a, getEleEnergies(a, RS_extrapolator), getEleEnergies(a, linear_extrapolator)) for a in alphas]
    
    return alpha_eleEnergies

def elements_displacements_of_alphas(ind, alphas, ele_ids):
    x = fv[ind].ravel()
    prob.setVars(x)
    d = opt.newton_step()
    
    # Linesearch prepare
    RS_extrapolator.linesearch_begin(x, d)
    linear_extrapolator.linesearch_begin(x, d)
    
    def getEleDisp(alpha, ex, ele_id):
        x_new = ex.linesearch_eval(alpha)
        # return x_new
        return x_new[m.elements()[ele_id]], x_new
    
    alpha_eleid_disps = []
    for ele_id in ele_ids:
        for a in alphas:
            rs_result, uv_ex = getEleDisp(a, RS_extrapolator, ele_id)
            F_true = RS_extrapolator.elementJacobian(ele_id, uv_ex.ravel())
            alpha_eleid_disps.append([a, ele_id, rs_result, uv_ex, RS_extrapolator.F_ex[ele_id], F_true, 
                                      getEleDisp(a, linear_extrapolator, ele_id)[0],  getEleDisp(a, linear_extrapolator, ele_id)[1]])
    return alpha_eleid_disps

In [ ]:
max_alpha = 2.5
alpha_step_size = 0.1
alphas = np.arange(0, max_alpha, alpha_step_size)

In [ ]:
alpha_eleEnergies = prepare_alpha_eleEnergies(vis_indx, alphas)

In [ ]:
plt.figure(figsize=(12, 6))
for alpha, rs_energy, linear_energy in alpha_eleEnergies:
    # if (alpha > 0.5): break
    print(alpha, np.argmax(rs_energy), np.max(rs_energy), np.argmax(linear_energy), np.max(linear_energy))
    plt.subplot(1, 2, 1)
    plt.scatter(np.arange(len(rs_energy)), rs_energy, label=f'rs, alpha={alpha : .1f}')
    plt.yscale('log')
    plt.ylim(top=1e-1)
    plt.subplot(1, 2, 2)
    plt.scatter(np.arange(len(linear_energy)), linear_energy, label=f'linear, alpha={alpha : .1f}')
    plt.yscale('log')
    plt.ylim(top=1e-1)
plt.legend()

In [ ]:
plt.figure(figsize=(12, 6))
d = alpha_eleEnergies
bad_element = 4097
plt.plot([a[0] for a in d], [a[1][bad_element] for a in d], label='rs')
plt.plot([a[0] for a in d], [a[2][bad_element] for a in d], label='linear')
plt.legend()

## Visualize Element(Tri) Displacements (Make Videos)

In [ ]:
vis_element_list = [bad_element]

In [ ]:
alpha_eleid_disps = elements_displacements_of_alphas(vis_indx, alphas, vis_element_list)

In [ ]:
alpha_eleid_disps[-1]

In [ ]:
curr_uv = fv[vis_indx]

In [ ]:
bad_el = mesh.Mesh(curr_uv[m.elements()[bad_element]], [[0, 1, 2]])

In [ ]:
vis_bad_element_field = [0] * m.numElements()
vis_bad_element_field[bad_element] = 1

In [ ]:

# bad_el_viewer = viewer.Viewer(bad_el, wireframe=True, offscreen=False)
# bad_el_viewer.update(preserveExisting=True, mesh=mesh.Mesh(curr_uv, m.elements()))

bad_el_viewer = viewer.Viewer(mesh.Mesh(curr_uv, m.elements()), 
                              scalarField={'data': vis_bad_element_field, 'colormap': matplotlib.cm.Blues}, 
                              wireframe=True, offscreen=False)
bad_el_viewer.show()

In [ ]:
# bad_el_viewer.setCameraParams(
# ((0.40787842551168507, -3.506662789296727, 15.529983851913853),
#  (0.0, 1.0, 0.0),
#  (0.40787842551168507, -3.506662789296727, 0.0))
# )
bad_el_viewer.setCameraParams(
((-0.345207632162704, 0.6286348083620463, 0.3091414236938079),
 (0.016971569601632562, 0.9910517532404719, -0.13239481947661108),
 (-0.377641805509965, 0.5792904938317247, -0.0643870002555588))
)

In [ ]:
import time

In [ ]:
# bad_el_viewer.recordStart(f'videos/{model_name}/invertedElement/{model_name}_LinearExtra_maxAlpha{max_alpha:.1f}.mp4', renderScale=16, 
#                           outputScale=4, framerate=30, lineWidthScale=0.3, transparentBackground=False)
# for alpha_ind, aed in enumerate(alpha_eleid_disps):
#     # bad_el.setVertices(aed[-2])
#     # bad_el_viewer.update(mesh=bad_el,  preserveExisting=False)
#     bad_el_viewer.update(mesh=mesh.Mesh(aed[-1], m.elements()), 
#                          scalarField={'data': vis_bad_element_field, 'colormap': matplotlib.cm.Blues})
#     time.sleep(0.1)
# bad_el_viewer.recordStop()

In [ ]:
[np.linalg.det(r[5]) for r in alpha_eleid_disps]

In [ ]:
# bad_el.setVertices(bad_el.vertices() + 1e-3)
bad_el_viewer.update()

## Functions for augmented alphas

In [ ]:
import video_writer

def writeVideo_plot_alphas(path, num_frames, plot_frame, plot_alpha_func, skipFrame=1, framerate=30):    
    from ipywidgets import IntProgress
    from IPython.display import display
    progress = IntProgress(min=0, max=num_frames)
    display(progress)
    axs = plot_frame(0)
    plt = plot_alpha_func(axs, 0)
    vw = video_writer.PlotVideoWriter(path, plt.gcf(), dpi=150, quality='-crf 10', tight_layout=False, framerate=framerate)
    plt.close()
    for frame in range(0, num_frames, skipFrame):
        axs = plot_frame(frame)
        plt = plot_alpha_func(axs, frame)
        vw.writeFrame(plt.gcf())
        plt.close()
        progress.value = frame + 1

In [ ]:
# writeVideo_plot_alphas(f'videos/cow/cow_init{initial_iter_num}iters_Poisson_brutal_linesearch.mp4', num_frames, ff, plot_cached_alphas)